# Controlled COVID-era training ablation
This standalone notebook preserves the original 2015–2017 notebook and artifacts. Those observations predate COVID; the historical test results cannot answer this question.

**Status: real-data ablation not run in the cloud.** Bundled data ends in 2017; the proxy rejected Yahoo history with HTTP 403. TensorFlow training is available. Synthetic integration tests validate mechanics only; they are not stock performance results.

Run in a fresh Google Colab CPU runtime. No clone, Drive mount or credentials required. Download real AAPL data or upload a CSV with Date/Close. Do not substitute synthetic prices for reported performance.

In [1]:
import importlib.util
if importlib.util.find_spec('tensorflow') is None or importlib.util.find_spec('yfinance') is None:
    %pip install -q tensorflow-cpu==2.18.1 pandas==2.2.3 numpy==2.0.2 scikit-learn==1.6.1 matplotlib==3.10.1 yfinance==0.2.65


## Predeclared protocol
A separate experiment: training 2015–2022, validation calendar 2023, test calendar 2024 (end exclusive 2025-01-01). This replaces neither the old dataset nor its original holdouts. All three regimes have exactly the same new validation/test target dates and inputs.

1. **with_2020**: all training years, same original LSTM architecture.
2. **exclude_2020**: omit every training window with ANY 2020 context or target row. Never splice the filtered series across the omitted year.
3. **post_2020**: train only on windows entirely within 2021–2022, requiring at least 252 training windows. This means post-2020, not a claim the pandemic ended in 2020.

Same 20-day lookback, architecture, optimizer, batch size, early-stopping budget and paired seeds 42/43/44. Fit a separate scaler only on each regime's eligible training rows. Pick checkpoints using validation loss; choose the regime by mean validation MSE in original price units. Save selection before testing. Report every fixed seed and its mean/std; do not pick the best seed. The test is evaluated once per frozen checkpoint. Do not change dates or settings after inspecting test outcomes.

Rolling one-step forecasts may use earlier realized held-out closes. No future target is in its own context. Source gaps over seven calendar days are rejected for investigation. Short missing sessions are not imputed; use complete daily trading data. Removing 2020 changes sample count and training distribution, so this is a sensitivity study, not causal proof of an effect of COVID volatility. Post-2020 may include pandemic effects. Adjusted prices reflect provider corporate-action revisions and are not a point-in-time data vintage.

In [2]:
"""Chronological one-step stock forecasting. No test-set model selection."""
import os
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')
os.environ.setdefault('TF_NUM_INTRAOP_THREADS', '2')
os.environ.setdefault('TF_NUM_INTEROP_THREADS', '2')
import argparse
import hashlib
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


def load_prices(path):
    df = pd.read_csv(path)
    df = df.rename(columns={'AAPL.Close': 'Close'})
    if not {'Date', 'Close'} <= set(df):
        raise ValueError('CSV must contain Date and Close columns')
    df = df[['Date', 'Close']].copy()
    df['Date'] = pd.to_datetime(df.Date, errors='raise')
    df['Close'] = pd.to_numeric(df.Close, errors='raise')
    if df.Date.duplicated().any() or df.isna().any().any():
        raise ValueError('Missing values and duplicate dates are not accepted')
    if not np.isfinite(df.Close).all() or (df.Close <= 0).any():
        raise ValueError('Prices must be finite and positive')
    return df.sort_values('Date').reset_index(drop=True)


def prepare(df, lookback=20):
    n = len(df)
    a, b = int(n * .7), int(n * .85)
    if lookback < 1 or a <= lookback or b-a < 2 or n-b < 2:
        raise ValueError('Insufficient observations for lookback and 70/15/15 split')
    values = df.Close.to_numpy().reshape(-1, 1)
    scaler = MinMaxScaler().fit(values[:a])
    scaled = scaler.transform(values).astype('float32')
    targets = np.arange(lookback, n)
    x = np.stack([scaled[i-lookback:i] for i in targets])
    y = scaled[targets]
    masks = {'train': targets < a, 'validation': (targets >= a) & (targets < b), 'test': targets >= b}
    splits = {name: (x[m], y[m], targets[m]) for name, m in masks.items()}
    return scaler, splits, (a, b)


def evaluate(actual, predicted, previous):
    actual, predicted, previous = map(lambda v: np.asarray(v).reshape(-1), (actual, predicted, previous))
    if not (len(actual) == len(predicted) == len(previous)) or len(actual) < 2:
        raise ValueError('Metrics require aligned arrays with at least two rows')
    if not all(np.isfinite(v).all() for v in (actual, predicted, previous)):
        raise ValueError('Metrics require finite values')
    mse = mean_squared_error(actual, predicted)
    return {'MSE': float(mse), 'RMSE': float(np.sqrt(mse)),
            'MAE': float(mean_absolute_error(actual, predicted)),
            'R2': float(r2_score(actual, predicted)),
            'DirectionalAccuracy': float(np.mean(np.sign(actual-previous) == np.sign(predicted-previous)))}


def build_model(lookback):
    import tensorflow as tf
    model = tf.keras.Sequential([tf.keras.Input((lookback, 1)),
        tf.keras.layers.LSTM(32, return_sequences=True),
        tf.keras.layers.LSTM(16), tf.keras.layers.Dense(1)])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=.001), loss='mse')
    return model


def dataset(x, y, batch_size=32):
    import tensorflow as tf
    ds = tf.data.Dataset.from_tensor_slices((x, y)).batch(batch_size)
    options = tf.data.Options()
    options.threading.private_threadpool_size = 1
    return ds.with_options(options)


def run(csv='data/aapl_history.csv', output='artifacts', epochs=40, lookback=20, ticker='AAPL', seed=42):
    import tensorflow as tf
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    if epochs < 1:
        raise ValueError('epochs must be positive')
    tf.keras.utils.set_random_seed(seed)
    tf.config.experimental.enable_op_determinism()
    df = load_prices(csv)
    scaler, splits, bounds = prepare(df, lookback)
    model = build_model(lookback)
    train, val = splits['train'], splits['validation']
    history = model.fit(dataset(*train[:2]), validation_data=dataset(*val[:2]), epochs=epochs, shuffle=False,
        callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)], verbose=0)
    out = Path(output)
    out.mkdir(parents=True, exist_ok=True)
    metric_rows, prediction_rows = [], []
    for split in ('validation', 'test'):
        x, _, idx = splits[split]
        prediction = scaler.inverse_transform(model(x, training=False).numpy()).ravel()
        actual = df.Close.to_numpy()[idx]
        previous = df.Close.to_numpy()[idx-1]
        for name, pred in [('LSTM', prediction), ('Naive', previous)]:
            metric_rows.append({'Split': split, 'Model': name, **evaluate(actual, pred, previous)})
        prediction_rows.append(pd.DataFrame({'Date': df.Date.iloc[idx].dt.strftime('%Y-%m-%d').to_numpy(),
            'Ticker': ticker, 'Split': split, 'Actual': actual, 'PreviousClose': previous,
            'LSTM': prediction, 'Naive': previous,
            'LSTMAbsoluteError': abs(actual-prediction), 'NaiveAbsoluteError': abs(actual-previous),
            'ActualDirection': np.sign(actual-previous), 'LSTMDirection': np.sign(prediction-previous),
            'NaiveDirection': np.zeros(len(idx), dtype=int)}))
    metrics = pd.DataFrame(metric_rows)
    predictions = pd.concat(prediction_rows, ignore_index=True)
    metrics.to_csv(out/'metrics.csv', index=False)
    predictions.to_csv(out/'power_bi_predictions.csv', index=False)
    pd.DataFrame(history.history).rename_axis('EpochZeroBased').to_csv(out/'training_history.csv')
    model.save(out/'model.keras')
    (out/'scaler.json').write_text(json.dumps({'min': scaler.min_.tolist(), 'scale': scaler.scale_.tolist(),
        'data_min': scaler.data_min_.tolist(), 'data_max': scaler.data_max_.tolist()} , indent=2))
    meta = {'ticker': ticker, 'seed': seed, 'lookback': lookback, 'requested_epochs': epochs,
        'completed_epochs': len(history.history['loss']), 'best_epoch': int(np.argmin(history.history['val_loss'])+1),
        'rows': len(df), 'split_boundaries': list(bounds),
        'date_range': [str(df.Date.min().date()), str(df.Date.max().date())],
        'sample_counts': {k: len(v[0]) for k,v in splits.items()},
        'data_sha256': hashlib.sha256(Path(csv).read_bytes()).hexdigest(),
        'tensorflow': tf.__version__, 'numpy': np.__version__,
        'protocol': 'Rolling one-step forecasts using observed previous closes; no recursive forecasts.'}
    (out/'run_metadata.json').write_text(json.dumps(meta, indent=2))
    fig, ax = plt.subplots(figsize=(10,4))
    ax.plot(df.Date, df.Close, label='Close')
    for i, label in zip(bounds, ['Validation starts', 'Test starts']): ax.axvline(df.Date.iloc[i], linestyle='--', label=label)
    ax.set(title=f'{ticker}: chronological split', ylabel='Price (historical USD)'); ax.legend(); fig.tight_layout(); fig.savefig(out/'data_split.png'); plt.close(fig)
    fig, ax = plt.subplots(figsize=(10,4))
    for k in ['loss','val_loss']: ax.plot(np.arange(1,len(history.history[k])+1),history.history[k], label=k)
    ax.set(xlabel='Epoch', ylabel='Scaled MSE', title='Learning curves'); ax.legend(); fig.tight_layout(); fig.savefig(out/'training.png'); plt.close(fig)
    test = predictions[predictions.Split == 'test']
    fig, ax = plt.subplots(figsize=(10,4))
    for k in ['Actual','LSTM','Naive']: ax.plot(pd.to_datetime(test.Date),test[k], label=k)
    ax.set(title='Untouched test period: one-step predictions', ylabel='Price (historical USD)'); ax.legend(); fig.tight_layout(); fig.savefig(out/'test_predictions.png'); plt.close(fig)
    fig, ax = plt.subplots(figsize=(10,4))
    for k in ['LSTM','Naive']: ax.hist(test.Actual-test[k], bins=20, alpha=.5, label=k)
    ax.set(title='Test residuals', xlabel='Actual minus predicted (USD)'); ax.legend(); fig.tight_layout(); fig.savefig(out/'residuals.png'); plt.close(fig)
    print(metrics.to_string(index=False))
    return metrics, predictions, meta




"""Predeclared COVID-era ablation; never overwrites the historical experiment."""
import argparse
import hashlib
import json
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler



@dataclass(frozen=True)
class Protocol:
    train_start: str = '2015-01-01'
    validation_start: str = '2023-01-01'
    test_start: str = '2024-01-01'
    test_end: str = '2025-01-01'  # exclusive
    post_start: str = '2021-01-01'  # post-2020, not a claim that COVID ended
    lookback: int = 20
    epochs: int = 40
    patience: int = 7
    seeds: tuple = (42, 43, 44)
    minimum_train_windows: int = 252


def prepare_experiment(df, p=Protocol()):
    dates = df.Date
    start, val, test, end = map(pd.Timestamp, (p.train_start, p.validation_start, p.test_start, p.test_end))
    if not start < val < test < end or not start < pd.Timestamp(p.post_start) < val:
        raise ValueError('Invalid chronological date boundaries')
    if p.lookback < 1 or p.epochs < 1 or p.patience < 1 or p.minimum_train_windows < 1:
        raise ValueError('Invalid training parameters')
    if not p.seeds or len(set(p.seeds)) != len(p.seeds):
        raise ValueError('Seeds must be nonempty and unique')
    df = df.loc[(dates >= start) & (dates < end)].reset_index(drop=True)
    if df.empty or not df.Date.is_monotonic_increasing or df.Date.duplicated().any():
        raise ValueError('Require sorted, unique observations')
    dates = df.Date
    # Refuse unexplained long source outages rather than silently joining them.
    if dates.diff().dt.days.dropna().gt(7).any():
        raise ValueError('Source gap over seven calendar days; supply complete daily trading history')
    if not (dates.dt.year == 2020).any() or not (dates < '2020-01-01').any():
        raise ValueError('COVID comparison requires pre-2020 and 2020 observations; bundled 2015–2017 data is insufficient')
    targets = np.arange(p.lookback, len(df))
    held = {
        'validation': targets[(dates.iloc[targets] >= val) & (dates.iloc[targets] < test)],
        'test': targets[(dates.iloc[targets] >= test) & (dates.iloc[targets] < end)],
    }
    for name, idx in held.items():
        if len(idx) < 2:
            raise ValueError(f'Insufficient {name} observations for fixed dates')
    if dates.min() > start + pd.Timedelta(days=7) or dates.max() < end - pd.Timedelta(days=7):
        raise ValueError('Data does not cover the predeclared experiment date range')
    training = dates < val
    masks = {
        'with_2020': training.to_numpy(),
        'exclude_2020': (training & (dates.dt.year != 2020)).to_numpy(),
        'post_2020': (training & (dates >= p.post_start)).to_numpy(),
    }
    values = df.Close.to_numpy().reshape(-1, 1)
    prepared, skipped = {}, {}
    for name, allowed in masks.items():
        # Build against original row positions, then require EVERY context/target row.
        # Filtering rows first would splice December 2019 to January 2021.
        idx = np.array([i for i in targets if allowed[i-p.lookback:i+1].all()], dtype=int)
        if len(idx) < p.minimum_train_windows:
            if name != 'post_2020':
                raise ValueError(f'{name}: need {p.minimum_train_windows} training windows, got {len(idx)}')
            skipped[name] = f'Only {len(idx)} training windows; minimum {p.minimum_train_windows}'
            continue
        scaler = MinMaxScaler().fit(values[allowed])
        scaled = scaler.transform(values).astype('float32')
        splits = {}
        for split, indices in {'train': idx, **held}.items():
            x = np.stack([scaled[i-p.lookback:i] for i in indices])
            splits[split] = (x, scaled[indices], indices)
        prepared[name] = (scaler, splits)
    return df, prepared, skipped


def run_experiment(csv, output, p=Protocol()):
    import tensorflow as tf
    df, prepared, skipped = prepare_experiment(load_prices(csv), p)
    out = Path(output)
    out.mkdir(parents=True, exist_ok=False)  # refuse replacing ANY earlier run
    protocol = {**asdict(p), 'data_sha256': hashlib.sha256(Path(csv).read_bytes()).hexdigest(),
                'tensorflow': tf.__version__, 'numpy': np.__version__, 'skipped': skipped,
                'selection': 'Lowest mean validation MSE in price units across fixed seeds; no test selection',
                'evaluation': 'Rolling one-step observed context; fixed validation/test dates for all variants',
                'architecture': 'LSTM(32,return_sequences=True),LSTM(16),Dense(1); Adam 0.001; batch 32; shuffle False'}
    (out/'protocol.json').write_text(json.dumps(protocol, indent=2))
    df.to_csv(out/'input_snapshot.csv', index=False)
    tf.config.experimental.enable_op_determinism()
    rows, predictions, models = [], [], []
    for regime, (scaler, splits) in prepared.items():
        for seed in p.seeds:
            tf.keras.backend.clear_session()
            tf.keras.utils.set_random_seed(seed)
            model = build_model(p.lookback)
            train, val = splits['train'], splits['validation']
            history = model.fit(dataset(*train[:2]), validation_data=dataset(*val[:2]),
                                epochs=p.epochs, shuffle=False, verbose=0,
                                callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=p.patience, restore_best_weights=True)])
            name = f'{regime}_seed{seed}'
            model.save(out/f'{name}.keras')
            pd.DataFrame(history.history).to_csv(out/f'{name}_history.csv', index=False)
            (out/f'{name}_scaler.json').write_text(json.dumps({
                'min': scaler.min_.tolist(), 'scale': scaler.scale_.tolist(),
                'data_min': scaler.data_min_.tolist(), 'data_max': scaler.data_max_.tolist(),
                'training_samples': len(train[2]), 'best_epoch': int(np.argmin(history.history['val_loss'])+1),
                'train_target_dates': df.Date.iloc[train[2]].dt.strftime('%Y-%m-%d').tolist()}, indent=2))
            idx = val[2]
            pred = scaler.inverse_transform(model(val[0], training=False).numpy()).ravel()
            actual, previous = df.Close.to_numpy()[idx], df.Close.to_numpy()[idx-1]
            rows.append({'Split': 'validation', 'Model': regime, 'Seed': seed, **evaluate(actual,pred,previous)})
            predictions.append(pd.DataFrame({'Date':df.Date.iloc[idx].to_numpy(), 'Split':'validation', 'Model':regime,
                                             'Seed':seed,'Actual':actual,'Prediction':pred,'PreviousClose':previous}))
            models.append((regime, seed, name))
    validation = pd.DataFrame(rows)
    winner = validation.groupby('Model').MSE.mean().sort_values().index[0]
    # Persist the selection BEFORE accessing test predictions, and never tune after it.
    (out/'selection.json').write_text(json.dumps({'selected_regime':winner,
        'mean_validation_mse':validation.groupby('Model').MSE.mean().to_dict(),
        'test_used_for_selection':False},indent=2))
    for regime, seed, name in models:
        scaler, splits = prepared[regime]
        x, _, idx = splits['test']
        model = tf.keras.models.load_model(out/f'{name}.keras')
        pred = scaler.inverse_transform(model(x,training=False).numpy()).ravel()
        actual, previous = df.Close.to_numpy()[idx], df.Close.to_numpy()[idx-1]
        rows.append({'Split':'test','Model':regime,'Seed':seed,**evaluate(actual,pred,previous)})
        predictions.append(pd.DataFrame({'Date':df.Date.iloc[idx].to_numpy(),'Split':'test','Model':regime,
                                         'Seed':seed,'Actual':actual,'Prediction':pred,'PreviousClose':previous}))
    common = next(iter(prepared.values()))[1]
    for split in ('validation','test'):
        idx = common[split][2]
        actual, previous = df.Close.to_numpy()[idx], df.Close.to_numpy()[idx-1]
        rows.append({'Split':split,'Model':'Naive','Seed':None,**evaluate(actual,previous,previous)})
        predictions.append(pd.DataFrame({'Date':df.Date.iloc[idx].to_numpy(),'Split':split,'Model':'Naive',
                                         'Seed':None,'Actual':actual,'Prediction':previous,'PreviousClose':previous}))
    metrics = pd.DataFrame(rows)
    metrics.to_csv(out/'metrics.csv',index=False)
    pd.concat(predictions,ignore_index=True).to_csv(out/'predictions.csv',index=False)
    keys = ['MSE','RMSE','MAE','R2','DirectionalAccuracy']
    summary = metrics.groupby(['Split','Model'])[keys].agg(['mean','std'])
    summary.to_csv(out/'summary.csv')
    print('Selected using validation only:',winner)
    print(summary.to_string())
    return metrics, winner




In [3]:
# Freeze these choices BEFORE training; use a NEW run directory each time.
PROTOCOL = Protocol()
CSV_PATH = None  # Set to an uploaded complete Date,Close CSV to avoid Yahoo.
DOWNLOAD_YAHOO = True
from datetime import datetime, timezone
OUTPUT = 'covid_runs/' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
print(asdict(PROTOCOL))


{'train_start': '2015-01-01', 'validation_start': '2023-01-01', 'test_start': '2024-01-01', 'test_end': '2025-01-01', 'post_start': '2021-01-01', 'lookback': 20, 'epochs': 40, 'patience': 7, 'seeds': (42, 43, 44), 'minimum_train_windows': 252}


## Real data acquisition and audit
Yahoo is optional and can rate limit. Failures are recorded explicitly without synthetic fallback. AAPL uses split/dividend-adjusted daily close consistently across the entire new experiment; never concatenate it with the original nominal CSV. A CSV upload should use a single consistent adjustment convention and asset. Record its provenance yourself. The run saves the input snapshot and SHA-256.

In [4]:
DATA_STATUS = 'not acquired'
if CSV_PATH is None and DOWNLOAD_YAHOO:
    try:
        import yfinance as yf
        raw = yf.download('AAPL', start=PROTOCOL.train_start, end=PROTOCOL.test_end,
                          auto_adjust=True, progress=False, threads=False)
        if raw.empty:
            raise RuntimeError('Yahoo returned no observations')
        close = raw['Close']
        if isinstance(close, pd.DataFrame): close = close.iloc[:, 0]
        folder = Path(OUTPUT)
        folder.parent.mkdir(parents=True, exist_ok=True)
        CSV_PATH = str(folder.parent / (folder.name + '_input.csv'))
        pd.DataFrame({'Date':close.index, 'Close':close.to_numpy()}).to_csv(CSV_PATH,index=False)
        Path(CSV_PATH + '.provenance.json').write_text(json.dumps({
            'provider':'Yahoo Finance via yfinance', 'ticker':'AAPL', 'auto_adjust':True,
            'retrieved_utc':datetime.now(timezone.utc).isoformat(),
            'requested_start':PROTOCOL.train_start,'requested_end_exclusive':PROTOCOL.test_end,
            'sha256':hashlib.sha256(Path(CSV_PATH).read_bytes()).hexdigest()},indent=2))
    except Exception as error:
        CSV_PATH = None
        DATA_STATUS = f'BLOCKED: {type(error).__name__}: {error}'
if CSV_PATH:
    # Fail loudly for bad/insufficient user data before any fitting.
    audit_df, audit_regimes, skipped = prepare_experiment(load_prices(CSV_PATH), PROTOCOL)
    DATA_STATUS = 'Ready for one predeclared experiment'
    print({name:{split:len(value[2]) for split,value in splits.items()}
           for name,(_,splits) in audit_regimes.items()})
    print('Skipped:', skipped)
print(DATA_STATUS)


{'with_2020': {'train': 1994, 'validation': 250, 'test': 252}, 'exclude_2020': {'train': 1721, 'validation': 250, 'test': 252}, 'post_2020': {'train': 483, 'validation': 250, 'test': 252}}
Skipped: {}
Ready for one predeclared experiment


In [5]:
if CSV_PATH:
    metrics, winner = run_experiment(CSV_PATH, OUTPUT, PROTOCOL)
    display(metrics)
    print('Validation-selected regime:', winner)
else:
    print('NOT RUN: provide real COVID-spanning data or retry on Colab with data access. No performance claims are available.')


Selected using validation only: post_2020
                                MSE                   RMSE                  MAE                  R2           DirectionalAccuracy          
                               mean         std       mean       std       mean       std      mean       std                mean       std
Split      Model                                                                                                                           
test       Naive           8.320328         NaN   2.884498       NaN   2.121529       NaN  0.987184       NaN            0.003968       NaN
           exclude_2020  978.671977  146.209402  31.224859  2.349518  25.809127  2.201528 -0.507501  0.225214            0.443122  0.011455
           post_2020      99.547138   62.553558   9.637525  3.161943   7.701083  2.421411  0.846662  0.096355            0.421958  0.002291
           with_2020     957.934575   90.207013  30.927433  1.463800  25.994487  1.271416 -0.475558  0.138951         

,Split,Model,Seed,MSE,RMSE,MAE,R2,DirectionalAccuracy
0,validation,with_2020,42.0,93.104915,9.649089,8.089297,0.687422,0.508000
1,validation,with_2020,43.0,109.922641,10.484400,8.742446,0.630961,0.508000
2,validation,with_2020,44.0,100.446045,10.022277,8.385561,0.662776,0.508000
3,validation,exclude_2020,42.0,79.454759,8.913740,7.196323,0.733250,0.544000
4,validation,exclude_2020,43.0,97.307986,9.864481,7.922555,0.673312,0.520000
5,validation,exclude_2020,44.0,108.678211,10.424884,8.365374,0.635139,0.492000
6,validation,post_2020,42.0,12.087282,3.476677,2.800852,0.959420,0.468000
7,validation,post_2020,43.0,13.475906,3.670954,3.021244,0.954758,0.476000
8,validation,post_2020,44.0,12.457337,3.529495,2.985885,0.958177,0.484000
9,test,with_2020,42.0,863.820527,29.390824,24.664332,-0.330589,0.428571


Validation-selected regime: post_2020


## Interpretation and export
MSE is price²; RMSE and MAE are price; R² is not accuracy. Directional accuracy compares sign(forecast − previous close) to sign(actual − previous close), treating zero separately. Naive previous-close forecasts predict zero movement, so their directional score is the unchanged-day frequency, not a 50% benchmark.

Compare every regime against the same naive forecasts. Use validation for selection, and test only as a final descriptive comparison. A lower error after dropping 2020 does not isolate volatility from reduced sample size, recency, price-level shifts, or seed variance. Three seeds quantify limited training variability, not a statistical significance test. If all variants lose to persistence, report it. Never copy synthetic test scores into the README as real-market evidence.

After a successful run, copy the printed real validation/test metrics and provenance to the README as a **new** results section, retaining the historical table. Archive output before any future study.

In [6]:
if CSV_PATH and Path(OUTPUT, 'metrics.csv').exists():
    import shutil
    archive = shutil.make_archive(OUTPUT, 'zip', OUTPUT)
    print('Saved', archive)
    try:
        from google.colab import files
        files.download(archive)
    except ImportError:
        pass


Saved /content/covid_runs/20261029T143050056902Z.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>